In [ ]:
from arize.experimental.datasets import ArizeDatasetsClient
from arize.experimental.datasets.utils.constants import GENERATIVE
import os
from dotenv import load_dotenv
import requests

load_dotenv()

HOST = os.getenv("ARIZE_HOST")
API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
DATASET_ID = "RGF0YXNldDo4OmdyWmY=" # taken from 02_dataset.ipynb

In [ ]:
import requests
import pandas as pd

def get_dataset(api_key: str, dataset_id: str):
    url = f"{HOST}/v2/datasets/{dataset_id}/examples?limit=50"
    headers = {
        "Authorization": f"Bearer {api_key}"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an exception for bad status codes
    data = response.json()
    return pd.DataFrame(data['examples'])

In [ ]:
df = get_dataset(API_KEY, DATASET_ID)
df

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# llm = ChatOpenAI(model="gpt-4.1-2025-04-14", temperature=0.0)
# chain = llm | StrOutputParser()

def sample_task(dataset_row) -> str:
  return dataset_row.get("label")
  # return chain.invoke(dataset_row.get("question"))

In [ ]:
sample_task(df.iloc[0])

In [ ]:
from typing import Any, Optional, Mapping
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    CodeEvaluator,
    JSONSerializable,
    TaskOutput
)

class SampleEvaluator(CodeEvaluator):
    def evaluate(
        self,
        *,
        dataset_row: Optional[Mapping[str, JSONSerializable]] = None,
        output: Optional[TaskOutput] = None,
        **_: Any,
    ) -> EvaluationResult:
        ground_truth = dataset_row.get("label") if dataset_row is not None else None
        task_output = output

        is_good = ground_truth == task_output
        label = "match expected result" if is_good else "does not match expected result"
        
        return EvaluationResult(
            score=float(is_good),
            label=label,
            explanation=f"ground truth: {ground_truth}, task output: {task_output}"
        )

In [ ]:
def run_experiment(df: pd.DataFrame, task_func, evaluator_instance):
    task_outputs = []
    eval_scores = []
    eval_labels = []
    eval_explanations = []

    for index, row in df.iterrows():
        # Execute the task function
        task_output = task_func(row)
        task_outputs.append(task_output)

        # Run the evaluator
        evaluation_result = evaluator_instance.evaluate(output=task_output, dataset_row=row)

        # Store the evaluation results
        eval_scores.append(evaluation_result.score)
        eval_labels.append(evaluation_result.label)
        eval_explanations.append(evaluation_result.explanation)

    # Create a new DataFrame with the results
    results_df = df.copy()
    results_df['output'] = task_outputs
    results_df['eval.accuracy.score'] = eval_scores
    results_df['eval.accuracy.label'] = eval_labels
    results_df['eval.accuracy.explanation'] = eval_explanations

    return results_df

In [ ]:
results_df = run_experiment(df, sample_task, SampleEvaluator())
results_df.head()

In [ ]:
import requests

def log_experiment(experiment_name: str, dataset_id: str, arize_api_key: str, results_df: pd.DataFrame, df: pd.DataFrame):
    experiment_runs = []

    for index, row in results_df.iterrows():
        original_row = df.loc[index] # Get the corresponding row from the original dataset
        experiment_run = {
            "exampleId": original_row["id"],
            "example_id": original_row["id"],
            "output": row["output"],
            "eval.accuracy.score": row["eval.accuracy.score"],
            "eval.accuracy.label": row["eval.accuracy.label"],
            "eval.accuracy.explanation": row["eval.accuracy.explanation"]

        }
        experiment_runs.append(experiment_run)

    # Construct the API request body
    request_body = {
        "name": experiment_name,
        "datasetId": dataset_id,
        "experimentRuns": experiment_runs
    }

    # Define the API endpoint
    url = f"{HOST}/v2/experiments"
    headers = {
        "Authorization": f"Bearer {arize_api_key}",
        "Content-Type": "application/json"
    }

    # Send the POST request
    try:
        response = requests.post(url, headers=headers, json=request_body)
        response.raise_for_status()  # Raise an exception for bad status codes
        print(f"API Response Status: {response.status_code}")
        print(f"API Response Body: {response.json()}")
    except requests.exceptions.RequestException as e:
        print(f"Error sending API request: {e}")
        if response is not None:
            print(f"Error Response Status: {response.status_code}")
            print(f"Error Response Body: {response.text}")

# Call the log_experiment function with appropriate arguments
experiment_name = "sample_v1"
log_experiment(experiment_name, DATASET_ID, API_KEY, results_df, df)
